# Will Betts - data exploration

Cleaned Comtrade exports (`data/cleaned/<year>clean.csv`, 2019-2023) and World Bank GDP (`data/archive/2024-fall/GDP_2023.csv`).

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import linalg as LA
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.linear_model import LinearRegression

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)
sns.set_theme(style="whitegrid")

In [3]:
REPO = Path.cwd().parent  # notebook lives in willbetts/
DATA = REPO / "data"
YEARS = range(2019, 2024)

# Cleaned exports by country x 18 industry groups (header "Precious Stones\t" has a trailing tab)
exports = {}
for y in YEARS:
    df = pd.read_csv(DATA / "cleaned" / f"{y}clean.csv")
    df.columns = df.columns.str.strip()
    exports[y] = df.set_index("Country")

# World Bank GDP, current US$, 1990-2023 (file starts with a BOM)
gdp = pd.read_csv(DATA / "archive" / "2024-fall" / "GDP_2023.csv", encoding="utf-8-sig")

exports[2023].head()

,Animals and By-products,"Vegetables, Fruits and Plants",Food and Preparations,Energy Drilling and Mining,Chemicals,Chemical Manufacturing,Raw Manufacturing,Wood Products,Clothing and Accessories,Textile,Metals,Machinery and Construction,Transportation,Weapons,Daily Instruments,Precious Stones,High-complexity Products,Services and Utilities
Country,,,,,,,,,,,,,,,,,,
Albania,"840,841,652.85","1,312,255,979.18","1,168,879,257.76","3,832,449,006.80","57,199,331.98","410,605,940.25","650,648,188.71","678,652,939.50","8,814,764,378.92","25,848,087.01","4,570,214,206.94","109,773,677.20","1,458,595,475.30","12,369,753.62","407,065,361.29","129,640,957.39","10,820,840,586.84","10,282,743,462.21"
Andorra,"760,403.19","2,007.56","2,172,680.17","300,989.01","36,547.06","3,881,437.88","3,618,370.55","14,088,608.69","15,948,289.00","418,620.87","5,271,191.90","44,310.81","104,813,760.97","284,103.14","33,174,255.40","7,603,242.43","62,031,837.58","372,005.37"
Angola,"78,181,904.85","75,937,624.73","122,021,509.23","70,317,238,719.79","5,696,086.17","14,577,462.40","30,598,294.75","84,385,962.21","5,976,276.98","1,896,689.60","232,166,604.58","2,321,414.57","106,002,162.29","2,239,235.80","4,307,803,536.33","3,224,834,876.72","428,951,637.75","62,538,314.75"
Antigua and Barbuda,"14,084,836.07","1,084,705.33","14,505,296.39","31,798,564.65","191,466.42","5,780,412.36","2,301,116.31","1,281,503.40","8,675,352.76","117,167.56","15,196,251.65","96,576.86","7,091,817.04","5,074.83","8,401,680.55","2,497,723.47","15,311,362.81",0.00
Argentina,"5,964,030,039.67","10,384,270,338.53","17,940,141,715.37","5,978,949,931.34","687,210,174.35","2,380,625,499.54","959,530,395.62","350,114,936.21","18,734,686.96","179,521,206.92","194,704,229.04","19,636,978.55","6,449,397,706.32","654,210.39","94,683,442.99","1,777,358,496.92","22,142,807,230.25","12,368,477,222.42"


## Does export diversification predict GDP growth?

1. Each country's export shares across the 18 industry groups (2019).
2. Diversification scores: entropy (higher = more diversified) and HHI (higher = more concentrated).
3. Merge with GDP and compute annualized growth 2019-2023.
4. Plot and regress growth on diversification, controlling for country size.

Diversification is measured in 2019 so it comes *before* the growth it is predicting.

In [4]:
# 1. Export shares: each row sums to 1
base = exports[2019]
shares = base.div(base.sum(axis=1), axis=0)

# 2. Diversification scores
entropy = -(shares * np.log(shares.where(shares > 0))).sum(axis=1)  # max = log(18) ~ 2.89
hhi = (shares ** 2).sum(axis=1)                                      # min = 1/18 ~ 0.056

div = pd.DataFrame({
    "entropy": entropy,
    "hhi": hhi,
    "top_industry": shares.idxmax(axis=1),
    "top_share": shares.max(axis=1),
    "total_exports": base.sum(axis=1),
})
div.sort_values("entropy").head(10)  # least diversified

,entropy,hhi,top_industry,top_share,total_exports
Country,,,,,
Yemen,0.21,0.93,"Vegetables, Fruits and Plants",0.96,"23,794,306.00"
Maldives,0.21,0.92,Animals and By-products,0.96,"158,011,440.00"
Angola,0.25,0.91,Energy Drilling and Mining,0.95,"34,840,084,948.00"
Libya,0.28,0.89,Energy Drilling and Mining,0.94,"29,339,360,227.00"
Azerbaijan,0.52,0.82,Energy Drilling and Mining,0.91,"19,719,432,246.00"
Botswana,0.55,0.81,Precious Stones,0.90,"5,341,608,483.00"
Montserrat,0.56,0.77,Energy Drilling and Mining,0.87,"5,699,711.00"
Brunei Darussalam,0.57,0.78,Energy Drilling and Mining,0.88,"7,302,296,525.00"
Kuwait,0.65,0.75,Energy Drilling and Mining,0.87,"67,771,388,681.00"


In [5]:
# 3. Comtrade and World Bank spell countries differently; map Comtrade -> World Bank
NAME_MAP = {
    "Bahamas": "Bahamas, The",
    "Bosnia Herzegovina": "Bosnia and Herzegovina",
    "China, Hong Kong Special Administrative Region": "Hong Kong SAR, China",
    "China, Macao Special Administrative Region": "Macao SAR, China",
    "Côte d'Ivoire": "Cote d'Ivoire",
    "Democratic Republic of the Congo": "Congo, Dem. Rep.",
    "Congo": "Congo, Rep.",
    "Egypt": "Egypt, Arab Rep.",
    "France, Monaco": "France",
    "Gambia": "Gambia, The",
    "Kyrgyzstan": "Kyrgyz Republic",
    "Lao People's Dem. Rep.": "Lao PDR",
    "Norway, Svalbard and Jan Mayen": "Norway",
    "Plurinational State of Bolivia": "Bolivia",
    "Republic of Korea": "Korea, Rep.",
    "Republic of Moldova": "Moldova",
    "Saint Lucia": "St. Lucia",
    "Saint Vincent and the Grenadines": "St. Vincent and the Grenadines",
    "Slovakia": "Slovak Republic",
    "State of Palestine": "West Bank and Gaza",
    "Swaziland": "Eswatini",
    "Switzerland, Liechtenstein": "Switzerland",
    "The Former Yugoslav Republic of Macedonia": "North Macedonia",
    "Turkey": "Turkiye",
    "USA, Puerto Rico and US Virgin Islands": "United States",
    "United Republic of Tanzania": "Tanzania",
}
# Expected to drop: EU-28, "Other Asia, nes", Montserrat (no World Bank row);
# Yemen, Lebanon, French Polynesia (no 2023 GDP); United States (missing from GDP_2023.csv)

g = gdp.set_index("Country Name")[["2019", "2023"]].rename(columns=lambda c: f"gdp_{c}")
df = div.rename(index=lambda c: NAME_MAP.get(c, c)).join(g, how="inner").dropna()

df["growth"] = (df["gdp_2023"] / df["gdp_2019"]) ** (1 / 4) - 1  # annualized, nominal US$
df["log_gdp"] = np.log(df["gdp_2019"])

print(f"{len(df)} of {len(div)} countries matched")
unmatched = sorted(set(div.rename(index=lambda c: NAME_MAP.get(c, c)).index) - set(df.index))
print("Unmatched:", unmatched)
df.describe()

154 of 161 countries matched
Unmatched: ['EU-28', 'French Polynesia', 'Lebanon', 'Montserrat', 'Other Asia, not elsewhere specified', 'United States', 'Yemen']


,entropy,hhi,top_share,total_exports,gdp_2019,gdp_2023,growth,log_gdp
count,154.00,154.00,154.00,154.00,154.00,154.00,154.00,154.00
mean,1.81,0.27,0.41,"126,930,088,378.54","413,941,813,168.27","488,020,864,594.79",0.05,24.78
std,0.52,0.18,0.19,"312,714,298,073.41","1,336,271,490,442.32","1,606,807,540,998.54",0.05,2.08
min,0.21,0.10,0.16,"9,914,614.00","427,425,039.70","678,976,265.00",-0.10,19.87
25%,1.57,0.15,0.28,"2,703,080,405.25","14,130,088,262.50","17,200,000,000.00",0.03,23.37
50%,1.93,0.21,0.35,"13,056,512,960.50","54,982,789,006.50","70,500,000,000.00",0.05,24.73
75%,2.20,0.31,0.48,"83,453,182,614.00","320,500,000,000.00","360,000,000,000.00",0.07,26.49
max,2.51,0.92,0.96,"2,693,005,021,777.00","14,300,000,000,000.00","17,800,000,000,000.00",0.35,30.29


In [6]:
# 4a. Plot: diversification vs growth, bubble size = GDP
fig = px.scatter(
    df.reset_index(names="country"),
    x="entropy", y="growth", size="gdp_2019", color="top_share",
    hover_name="country", hover_data=["top_industry"],
    trendline="ols", size_max=50,
    labels={"entropy": "Export diversification, 2019 (entropy)",
            "growth": "Annualized GDP growth 2019-2023 (nominal US$)",
            "top_share": "Largest industry share"},
)
fig.update_layout(yaxis_tickformat=".0%")
fig.show()

In [7]:
# 4b. Regressions. Model 2 controls for size (big economies are both more diversified and slower-growing)
m1 = smf.ols("growth ~ entropy", data=df).fit(cov_type="HC1")
m2 = smf.ols("growth ~ entropy + log_gdp", data=df).fit(cov_type="HC1")
print(m1.summary().tables[1])
print(m2.summary().tables[1])
print(f"R^2: {m1.rsquared:.3f} vs {m2.rsquared:.3f}")

                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.0356      0.016      2.240      0.025       0.004       0.067
entropy        0.0090      0.008      1.146      0.252      -0.006       0.024
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.1095      0.048      2.284      0.022       0.016       0.204
entropy        0.0134      0.009      1.566      0.117      -0.003       0.030
log_gdp       -0.0033      0.002     -1.593      0.111      -0.007       0.001
R^2: 0.010 vs 0.029


In [8]:
# Outliers worth a look: growth is in nominal US$, so inflation and exchange-rate swings show up here
df.sort_values("growth")[["top_industry", "top_share", "entropy", "growth"]].iloc[np.r_[0:5, -5:0]]

,top_industry,top_share,entropy,growth
Libya,Energy Drilling and Mining,0.94,0.28,-0.10
Nigeria,Energy Drilling and Mining,0.84,0.71,-0.06
Japan,High-complexity Products,0.40,1.81,-0.05
"Macao SAR, China",Transportation,0.23,2.00,-0.05
Lao PDR,Energy Drilling and Mining,0.35,2.23,-0.04
Uzbekistan,Precious Stones,0.32,2.11,0.14
Ethiopia,"Vegetables, Fruits and Plants",0.56,1.54,0.14
Georgia,High-complexity Products,0.23,2.20,0.15
Armenia,Energy Drilling and Mining,0.28,2.01,0.15
Guyana,Precious Stones,0.53,1.62,0.35


### Things to try next
- Swap `entropy` for `hhi` (expect the opposite sign).
- Drop countries whose top industry is `Energy Drilling and Mining`: the 2019-2023 oil price swing drives their growth.
- Use 2023 diversification, or the *change* in entropy 2019-2023, as the predictor.
- Repeat with a different window (e.g. 2019-2021 for the COVID shock).